# Gold — dim_channel_scd1 (SCD 1, poc method)

One row per `channel_id`, latest values.
Method (poc `scd_type_1`, helper `_scd`): **changed keys** in the window → re-read their full current rows → metadata `etl_date` / `etl_by` → `add_checksums` → **full**: overwrite / **incremental**: MERGE on `primary_checksum`, update only when `record_checksum` differs → Unknown row → grain check. Widgets: `load_type`, `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). Runs in task `gold_dims_scd1` (nothing depends on it); the live dim is unchanged.

**Changed keys:** channels read (`stg_channels.snapshot_date`) in the window. Values from `stg_channels_latest`. `created_at` comes from that row: checked on dev 2026-11-04, no channel has more than one `created_at`, so no MIN over history is needed.

`channel_key = xxhash64(channel_id)`: same key as `dim_channel`, so this table can replace it without touching the facts. `created_at` / `modified_at` (= `snapshot_ts`) are not hashed: a re-read with the same values writes nothing.

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow (today's reads included)
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
FULL = LOAD_TYPE == "full"
print("full: every key" if FULL else f"incremental: keys changed in [{FROM_DATE}, {TO_DATE}) (Vietnam days)")

In [ ]:
keys = (None if FULL else changed_keys(
    spark.table(f"{SILVER}.stg_channels").where(in_window(F.col("snapshot_date"), FROM_DATE, TO_DATE)).select("channel_id")))
latest = spark.table(f"{SILVER}.stg_channels_latest")
if keys is not None:
    latest = latest.join(keys, "channel_id", "left_semi")
src = (latest
       .select(F.xxhash64("channel_id").alias("channel_key"), "channel_id", "channel_title", "channel_description",
               "custom_url", "country", "default_language", "thumbnail_url", "published_at", "created_at",
               F.col("snapshot_ts").alias("modified_at"))
       .withColumn("etl_date", F.current_timestamp())
       .withColumn("etl_by", F.lit("Databricks")))
src = add_checksums(src, ["channel_id"], exclude=["channel_key", "created_at", "modified_at"])

In [ ]:
load_scd1(src, f"{GOLD}.dim_channel_scd1", """
        channel_key BIGINT, channel_id STRING, channel_title STRING, channel_description STRING, custom_url STRING,
        country STRING, default_language STRING, thumbnail_url STRING, published_at TIMESTAMP,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING""",
    "Channel dimension, SCD 1 (poc checksum method). channel_key = xxhash64(channel_id), -1 = Unknown.",
    ["channel_id"], LOAD_TYPE, {"channel_key": "-1", "channel_title": "'Unknown'", "etl_date": "current_timestamp()", "etl_by": "'Databricks'"})